In [ ]:
# In[1]:

import pandas as pd

# Load telemetry CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Ensure timestamp columns parsed as UTC datetimes (Unix seconds)
df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['timestamp_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['timestamp_dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_error['timestamp_dt'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregates per (cmdb_id, kpi_name)
# Ensure value numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

metric_ts_min = df_metric['timestamp_dt'].min()
metric_ts_max = df_metric['timestamp_dt'].max()

metric_agg = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    count=('value', 'count'),
    ts_min=('timestamp_dt', 'min'),
    ts_max=('timestamp_dt', 'max'),
    value_min=('value', 'min'),
    value_max=('value', 'max'),
    value_mean=('value', 'mean'),
    p50=('value', lambda x: x.quantile(0.5)),
    p90=('value', lambda x: x.quantile(0.9)),
    p95=('value', lambda x: x.quantile(0.95))
)

metric_agg_top50 = metric_agg.sort_values('count', ascending=False).head(50)

# 2) trace.csv: unique trace_name with count, p50, p95 and timestamp min/max
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
trace_ts_min = df_trace['timestamp_dt'].min()
trace_ts_max = df_trace['timestamp_dt'].max()

trace_agg = df_trace.groupby('trace_name', as_index=False).agg(
    count=('value', 'count'),
    p50=('value', lambda x: x.quantile(0.5)),
    p95=('value', lambda x: x.quantile(0.95))
).sort_values('count', ascending=False)

# 3) log.csv: unique log_name with count, p50, p95 and timestamp min/max
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
log_ts_min = df_log['timestamp_dt'].min()
log_ts_max = df_log['timestamp_dt'].max()

log_agg = df_log.groupby('log_name', as_index=False).agg(
    count=('value', 'count'),
    p50=('value', lambda x: x.quantile(0.5)),
    p95=('value', lambda x: x.quantile(0.95))
).sort_values('count', ascending=False)

# 4) error_logs.csv: total row count and top 5 rows for inspection
error_count = len(df_error)
error_top5 = df_error[['timestamp_dt', 'cmdb_id', 'message']].sort_values('timestamp_dt').head(5)

# Compact outputs (do not display full raw series)
metric_agg_top50, (metric_ts_min, metric_ts_max), trace_agg, (trace_ts_min, trace_ts_max), log_agg, (log_ts_min, log_ts_max), error_count, error_top5

```
Out[1]:
```
summary = (
    "Telemetry summary (UTC):\n"
    "- Time range for all telemetry: 2024-01-17 06:59:00 → 2024-01-17 07:23:00 UTC.\n"
    "- Metrics: the top reporting component is 'adservice' (many KPIs: cpu, diskio, latency, mem). Other active services in the top results include productcatalogservice and paymentservice.\n"
    "- Traces: several trace series show high counts and tail latency. Examples: trace.from_root.row_count has high p95 (~820) and some to_frontendservice row_count p95 values reach thousands; some duration p95 values reach ~0.59s for frontend-related traces.\n"
    "- Logs: log.error_count has 229 samples with p95 = 0.0 and log.row_count exists (p95 large), indicating no elevated normalized error_count in logs during this window.\n"
    "- Error logs file: zero rows (no raw error log entries found).\n"
    "Interpretation & next steps:\n"
    "- There are no explicit error log entries and normalized log.error_count is zero, so start investigation using metrics and traces.\n"
    "- Focus first on adservice (high telemetry presence) and frontend-related trace paths (high p95 durations / row_count spikes) to see if latency or load correlates with user-facing issues.\n"
    "- Slice traces and relevant KPIs (CPU, memory, disk I/O, latency KPIs) around the highest trace row_count / duration p95 timepoints to pinpoint which service and KPI correlate with the observed tail latencies."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count                    ts_min                    ts_max     value_min     value_max    value_mean           p50           p90           p95
0               adservice         cpu     25 2024-01-17 06:59:00+00:00 2024-01-17 07:23:00+00:00  6.348562e-01  1.047275e+00  7.320195e-01  7.221734e-01  7.785716e-01  7.952514e-01
1               adservice      diskio     25 2024-01-17 06:59:00+00:00 2024-01-17 07:23:00+00:00  5.061264e+04  1.710623e+05  7.012718e+04  6.040297e+04  8.764588e+04  1.102780e+05
2               adservice  latency-50     25 2024-01-17 06:59:00+00:00 2024-01-17 07:23:00+00:00  2.980309e-03  3.011296e-03  2.999620e-03  3.000467e-03  3.006093e-03  3.009481e-03
3               adservice  latency-90     25 2024-01-17 06:59:00+00:00 2024-01-17 07:23:00+00:00  4.599163e-03  4.630311e-03  4.613011e-03  4.611327e-03  4.620803e-03  4.627493e-03
4               adservice         mem     25 2024-01-17 06:59:00+00:00 2024-01-17 07:23:00+00:00  1.092291e+08  1.113270e+08  1.094733e+08  1.093913e+08  1.095261e+08  1.095685e+08
..                    ...         ...    ...                       ...                       ...           ...           ...           ...           ...           ...           ...
45         paymentservice         mem     25 2024-01-17 06:59:00+00:00 2024-01-17 07:23:00+00:00  3.990787e+07  4.115695e+07  4.041567e+07  4.039653e+07  4.086942e+07  4.105066e+07
46         paymentservice      socket     25 2024-01-17 06:59:00+00:00 2024-01-17 07:23:00+00:00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
47         paymentservice    workload     25 2024-01-17 06:59:00+00:00 2024-01-17 07:23:00+00:00  6.144833e-01  1.161133e+00  9.273487e-01  9.557333e-01  1.077157e+00  1.105930e+00
48  productcatalogservice         cpu     25 2024-01-17 06:59:00+00:00 2024-01-17 07:23:00+00:00  2.982385e+00  3.766162e+00  3.495538e+00  3.528902e+00  3.660671e+00  3.707133e+00
49  productcatalogservice  latency-50     25 2024-01-17 06:59:00+00:00 2024-01-17 07:23:00+00:00  1.800325e-03  2.083761e-03  1.922202e-03  1.925702e-03  1.993192e-03  2.014825e-03

[50 rows x 11 columns], (Timestamp('2024-01-17 06:59:00+0000', tz='UTC'), Timestamp('2024-01-17 07:23:00+0000', tz='UTC')),                                   trace_name  count          p50          p95
15                 trace.from_root.row_count    175    24.000000   819.900000
14                trace.from_root.error_rate    175     0.000000     0.000000
13              trace.from_root.duration_p95    175     0.000122     0.265500
12             trace.from_root.duration_mean    175     0.000069     0.099540
4   trace.from_frontendservice.duration_mean    125     0.003881     0.593594
..                                       ...    ...          ...          ...
32     trace.to_paymentservice.duration_mean     50     0.000127     0.000263
31        trace.to_frontendservice.row_count     50  1387.000000  8015.350000
30       trace.to_frontendservice.error_rate     50     0.000000     0.000000
29     trace.to_frontendservice.duration_p95     50     0.119268     0.302611
28    trace.to_frontendservice.duration_mean     50     0.040965     0.108212

[44 rows x 4 columns], (Timestamp('2024-01-17 06:59:00+0000', tz='UTC'), Timestamp('2024-01-17 07:23:00+0000', tz='UTC')),           log_name  count    p50     p95
0  log.error_count    229    0.0     0.0
1    log.row_count    229  456.0  2645.6, (Timestamp('2024-01-17 06:59:00+0000', tz='UTC'), Timestamp('2024-01-17 07:23:00+0000', tz='UTC')), 0, Empty DataFrame
Columns: [timestamp_dt, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse df_metric if already in session; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp parsed as UTC datetimes and value numeric
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
else:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp_dt'], utc=True)

df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Candidate components
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Components with zero metric rows in the entire metric.csv
present_components = df_metric['cmdb_id'].unique().tolist()
components_with_zero_metric_rows = [c for c in candidates if c not in present_components]

# 2) Compute global thresholds (p95 and p05) across the full metric.csv series for candidate components
df_cand_full = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

if df_cand_full.empty:
    thresholds = pd.DataFrame(columns=['cmdb_id','kpi_name','p05','p95'])
else:
    thresholds = (
        df_cand_full
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            p95=('value', lambda x: x.quantile(0.95)),
            p05=('value', lambda x: x.quantile(0.05))
        )
    )

# 3) Filter metric.csv rows to the incident window (UTC)
window_start = pd.Timestamp('2024-01-17 06:56:45', tz='UTC')
window_end = pd.Timestamp('2024-01-17 07:26:45', tz='UTC')

df_window = df_metric[
    (df_metric['timestamp_dt'] >= window_start) &
    (df_metric['timestamp_dt'] <= window_end) &
    (df_metric['cmdb_id'].isin(candidates))
].copy()

# If there are no rows in window, prepare empty result structure
if df_window.empty:
    result_compact = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points_in_window','count_above_p95',
        'count_below_p05','total_anomalies','earliest_anomaly_timestamp','p05','p95','sample_anomalies'
    ])
    result_compact.index.name = None
else:
    # 4) Merge thresholds into windowed rows so we can mark anomalies
    df_win_thr = pd.merge(df_window, thresholds, on=['cmdb_id', 'kpi_name'], how='left')

    # Determine anomaly direction for each row
    def anomaly_label(row):
        if pd.isna(row.get('p95')) or pd.isna(row.get('p05')):
            return None
        if row['value'] > row['p95']:
            return 'ABOVE'
        if row['value'] < row['p05']:
            return 'BELOW'
        return None

    df_win_thr['anomaly'] = df_win_thr.apply(anomaly_label, axis=1)

    # Group by (cmdb_id, kpi_name)
    grouped = df_win_thr.groupby(['cmdb_id', 'kpi_name'], sort=False)

    # total points in window
    agg_pts = grouped.size().rename('total_points_in_window').reset_index()

    # counts above/below
    count_above = grouped.apply(lambda g: (g['value'] > g['p95']).sum()).rename('count_above_p95').reset_index()
    count_below = grouped.apply(lambda g: (g['value'] < g['p05']).sum()).rename('count_below_p05').reset_index()

    # earliest anomaly timestamp per group
    def earliest_anom_ts(g):
        an = g[g['anomaly'].notna()]
        if an.empty:
            return pd.NaT
        return an['timestamp_dt'].min()
    earliest = grouped.apply(earliest_anom_ts).rename('earliest_anomaly_timestamp').reset_index()

    # sample anomalies up to 10 rows formatted compactly
    def sample_anomalies_str(g, max_samples=10):
        an = g[g['anomaly'].notna()].sort_values('timestamp_dt').head(max_samples)
        if an.empty:
            return ''
        parts = an.apply(lambda r: f"{r['timestamp_dt'].isoformat()}|{r['value']:.6g}|{r['anomaly']}", axis=1).tolist()
        return "; ".join(parts)
    samples = grouped.apply(lambda g: sample_anomalies_str(g)).rename('sample_anomalies').reset_index()

    # Merge aggregates
    result = agg_pts.merge(count_above, on=['cmdb_id','kpi_name']).merge(count_below, on=['cmdb_id','kpi_name']).merge(earliest, on=['cmdb_id','kpi_name']).merge(samples, on=['cmdb_id','kpi_name'])

    # Attach thresholds (p05,p95)
    result = pd.merge(result, thresholds, on=['cmdb_id','kpi_name'], how='left')

    # total anomalies and sort
    result['total_anomalies'] = result['count_above_p95'] + result['count_below_p05']
    result_sorted = result.sort_values('total_anomalies', ascending=False).reset_index(drop=True)

    # Compact columns and limit to top 50 rows
    result_compact = result_sorted[['cmdb_id','kpi_name','total_points_in_window','count_above_p95','count_below_p05','total_anomalies','earliest_anomaly_timestamp','p05','p95','sample_anomalies']].head(50)

# Final outputs: compact result table and components with zero metric rows
result_compact, components_with_zero_metric_rows

```
Out[2]:
```
Summary (plain English):

- All five candidate components have metric data in metric.csv (no candidate is missing metrics).

- Main finding: checkoutservice shows the most anomalies in the incident window. Multiple KPIs for checkoutservice (cpu, latency-50, latency-90, mem, workload, etc.) each have 4 anomalies in the window; earliest anomaly timestamps are around 2024-01-17T06:59:00+00:00–07:03:00+00:00. The per-row samples show both ABOVE and BELOW threshold events (i.e., values crossing the computed p95 or p05).

- Other candidates show few or no anomalies for many KPIs (examples shown: currencyservice.error, emailservice.diskio/socket, productcatalogservice.socket, recommendationservice.diskio all had 0 anomalies in the window).

- Counts: the aggregated table contains 33 (cmdb_id, kpi_name) groups for these candidates; results are sorted by total anomalies (top rows are checkoutservice KPIs).

Recommended next steps:
1. Prioritize investigation of checkoutservice around the earliest anomaly timestamps (06:59–07:03 UTC): correlate its anomalous KPI samples with traces and any logs.
2. Inspect whether the anomalies are spikes (above p95) or drops (below p05) for each KPI and whether they align with user-facing trace latency increases.
3. If confirmed, drill into host/container metrics and traces for checkoutservice at those timestamps to identify root cause (CPU, memory, I/O, or network).

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  total_points_in_window  count_above_p95  count_below_p05  total_anomalies earliest_anomaly_timestamp           p05           p95                                   sample_anomalies
0         checkoutservice         cpu                      25                2                2                4  2024-01-17 06:59:00+00:00  3.451207e-01  4.609336e-01  2024-01-17T06:59:00+00:00|0.320249|BELOW; 2024...
1         checkoutservice  latency-50                      25                2                2                4  2024-01-17 07:03:00+00:00  1.294136e-01  4.579993e-01  2024-01-17T07:03:00+00:00|0.0979062|BELOW; 202...
2         checkoutservice  latency-90                      25                2                2                4  2024-01-17 07:03:00+00:00  2.281387e-01  2.053321e+00  2024-01-17T07:03:00+00:00|0.217798|BELOW; 2024...
3         checkoutservice         mem                      25                2                2                4  2024-01-17 07:02:00+00:00  1.192098e+07  1.216699e+07  2024-01-17T07:02:00+00:00|1.19089e+07|BELOW; 2...
4         checkoutservice    workload                      25                2                2                4  2024-01-17 06:59:00+00:00  7.041967e-01  1.089620e+00  2024-01-17T06:59:00+00:00|0.7022|BELOW; 2024-0...
..                    ...         ...                     ...              ...              ...              ...                        ...           ...           ...                                                ...
28        currencyservice       error                      25                0                0                0                        NaT  0.000000e+00  0.000000e+00                                                   
29           emailservice      diskio                      25                0                0                0                        NaT  0.000000e+00  0.000000e+00                                                   
30           emailservice      socket                      25                0                0                0                        NaT  3.000000e+00  3.000000e+00                                                   
31  productcatalogservice      socket                      25                0                0                0                        NaT  4.000000e+00  4.000000e+00                                                   
32  recommendationservice      diskio                      25                0                0                0                        NaT  0.000000e+00  0.000000e+00                                                   

[33 rows x 10 columns], [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp parsed as UTC datetimes and value numeric
df_trace['timestamp_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Identify trace_name values that include 'checkoutservice' (case-insensitive)
substr = 'checkoutservice'
mask = df_trace['trace_name'].str.contains(substr, case=False, na=False)
trace_names_found = df_trace.loc[mask, 'trace_name'].unique().tolist()
identified_trace_names = pd.DataFrame({'trace_name': sorted(trace_names_found)})

# Compute global thresholds (p05, p95) across full trace.csv for each identified trace_name
if identified_trace_names.empty:
    thresholds = pd.DataFrame(columns=['trace_name','p05','p95'])
else:
    thresholds = (
        df_trace[df_trace['trace_name'].isin(trace_names_found)]
        .groupby('trace_name', as_index=False)
        .agg(
            p95=('value', lambda x: x.quantile(0.95)),
            p05=('value', lambda x: x.quantile(0.05))
        )
    )

# 2) Filter trace.csv to the incident window
window_start = pd.Timestamp('2024-01-17 06:56:45', tz='UTC')
window_end = pd.Timestamp('2024-01-17 07:26:45', tz='UTC')

df_trace_win = df_trace[
    (df_trace['timestamp_dt'] >= window_start) &
    (df_trace['timestamp_dt'] <= window_end) &
    (df_trace['trace_name'].isin(trace_names_found))
].copy()

# Prepare empty result if no data
if df_trace_win.empty:
    result_compact = pd.DataFrame(columns=[
        'trace_name','total_points_in_window','count_above_p95','count_below_p05',
        'total_anomalies','earliest_anomaly_timestamp','p05','p95','sample_anomalies'
    ])
else:
    # Merge thresholds into windowed rows
    df_tw = pd.merge(df_trace_win, thresholds, on='trace_name', how='left')

    # Label anomalies
    def an_label(r):
        if pd.isna(r['p95']) or pd.isna(r['p05']):
            return None
        if r['value'] > r['p95']:
            return 'ABOVE'
        if r['value'] < r['p05']:
            return 'BELOW'
        return None
    df_tw['anomaly'] = df_tw.apply(an_label, axis=1)

    grp = df_tw.groupby('trace_name', sort=False)

    # Aggregates
    agg_pts = grp.size().rename('total_points_in_window').reset_index()
    count_above = grp.apply(lambda g: (g['value'] > g['p95']).sum()).rename('count_above_p95').reset_index()
    count_below = grp.apply(lambda g: (g['value'] < g['p05']).sum()).rename('count_below_p05').reset_index()

    def earliest_anom(g):
        an = g[g['anomaly'].notna()]
        if an.empty:
            return pd.NaT
        return an['timestamp_dt'].min()
    earliest = grp.apply(earliest_anom).rename('earliest_anomaly_timestamp').reset_index()

    def sample_anoms(g, max_samples=10):
        an = g[g['anomaly'].notna()].sort_values('timestamp_dt').head(max_samples)
        if an.empty:
            return ''
        s = an.apply(lambda r: f"{r['timestamp_dt'].isoformat()}|{r['cmdb_id']}|{r['value']:.6g}|{r['anomaly']}", axis=1)
        return "; ".join(s.tolist())
    samples = grp.apply(lambda g: sample_anoms(g)).rename('sample_anomalies').reset_index()

    # Merge and attach thresholds
    result = agg_pts.merge(count_above, on='trace_name').merge(count_below, on='trace_name').merge(earliest, on='trace_name').merge(samples, on='trace_name')
    result = pd.merge(result, thresholds, on='trace_name', how='left')

    result['total_anomalies'] = result['count_above_p95'] + result['count_below_p05']
    result_compact = result.sort_values('total_anomalies', ascending=False).reset_index(drop=True)
    # select and limit columns
    result_compact = result_compact[['trace_name','total_points_in_window','count_above_p95','count_below_p05','total_anomalies','earliest_anomaly_timestamp','p05','p95','sample_anomalies']].head(20)

# Final outputs: identified trace names, thresholds, and compact anomaly table
identified_trace_names, thresholds, result_compact

```
Out[3]:
```
Summary (plain English):

- Found 8 checkoutservice-related trace series:
  - trace.from_checkoutservice.{duration_mean, duration_p95, error_rate, row_count}
  - trace.to_checkoutservice.{duration_mean, duration_p95, error_rate, row_count}

- Global thresholds (full trace.csv):
  - from_checkoutservice.duration_mean: p95 ≈ 0.06697, p05 ≈ 0.000015
  - from_checkoutservice.duration_p95: p95 ≈ 0.31108, p05 ≈ 0.000022
  - from_checkoutservice.error_rate: p95 = p05 = 0
  - from_checkoutservice.row_count: p95 = 255.8, p05 = 20
  - to_checkoutservice.duration_mean: p95 ≈ 0.61988, p05 ≈ 0.003616
  - to_checkoutservice.duration_p95: p95 ≈ 1.63524, p05 ≈ 0.005657
  - to_checkoutservice.error_rate: p95 = p05 = 0
  - to_checkoutservice.row_count: p95 = 260.7, p05 = 23

- Incident-window (2024-01-17 06:56:45 → 07:26:45 UTC) anomaly summary:
  - Highest anomaly counts:
    - trace.from_checkoutservice.duration_mean: 125 points, 7 above p95, 7 below p05 (14 total); earliest anomaly 2024-01-17 06:59:00 UTC.
    - trace.from_checkoutservice.duration_p95: 125 points, 7 above p95, 7 below p05 (14 total); earliest anomaly 2024-01-17 06:59:00 UTC.
    - trace.from_checkoutservice.row_count: 125 points, 7 above p95, 5 below p05 (12 total); earliest anomaly 2024-01-17 06:59:00 UTC.
  - to_checkoutservice series also show anomalies:
    - duration_mean: 75 points, 4 above, 4 below (8 total); earliest anomaly 2024-01-17 07:01:00 UTC.
    - duration_p95: 75 points, 4 above, 4 below (8 total); earliest anomaly 2024-01-17 07:01:00 UTC.
    - row_count: 75 points, 4 above, 3 below (7 total); earliest anomaly 2024-01-17 06:59:00 UTC.
  - error_rate series (both to/from) had no anomalies (all zeros).

- Sample anomaly observations (from returned rows): anomalies include both ABOVE and BELOW threshold events and involve various caller/callee services (e.g., productcatalogservice, emailservice, frontendservice) interacting with checkoutservice during the anomaly times.

Interpretation and next steps:
- Checkoutservice-related traces show clear abnormalities starting around 06:59 UTC (with some to_checkoutservice latencies showing earliest anomalies near 07:01 UTC). Both spikes and drops relative to historical thresholds are present.
- Prioritize investigating checkoutservice around 06:59–07:03 UTC: correlate these trace anomalies with checkoutservice metric anomalies (CPU, memory, workload) and any available logs to determine whether latency, traffic spikes, or upstream/downstream issues caused the observed behavior.

The original code execution output of IPython Kernel is also provided below for reference:

(                                 trace_name
0  trace.from_checkoutservice.duration_mean
1   trace.from_checkoutservice.duration_p95
2     trace.from_checkoutservice.error_rate
3      trace.from_checkoutservice.row_count
4    trace.to_checkoutservice.duration_mean
5     trace.to_checkoutservice.duration_p95
6       trace.to_checkoutservice.error_rate
7        trace.to_checkoutservice.row_count,                                  trace_name         p95        p05
0  trace.from_checkoutservice.duration_mean    0.066966   0.000015
1   trace.from_checkoutservice.duration_p95    0.311075   0.000022
2     trace.from_checkoutservice.error_rate    0.000000   0.000000
3      trace.from_checkoutservice.row_count  255.800000  20.000000
4    trace.to_checkoutservice.duration_mean    0.619875   0.003616
5     trace.to_checkoutservice.duration_p95    1.635244   0.005657
6       trace.to_checkoutservice.error_rate    0.000000   0.000000
7        trace.to_checkoutservice.row_count  260.700000  23.000000,                                  trace_name  total_points_in_window  count_above_p95  count_below_p05  total_anomalies earliest_anomaly_timestamp        p05         p95                                   sample_anomalies
0  trace.from_checkoutservice.duration_mean                     125                7                7               14  2024-01-17 06:59:00+00:00   0.000015    0.066966  2024-01-17T06:59:00+00:00|productcatalogservic...
1   trace.from_checkoutservice.duration_p95                     125                7                7               14  2024-01-17 06:59:00+00:00   0.000022    0.311075  2024-01-17T06:59:00+00:00|productcatalogservic...
2      trace.from_checkoutservice.row_count                     125                7                5               12  2024-01-17 06:59:00+00:00  20.000000  255.800000  2024-01-17T06:59:00+00:00|emailservice|6|BELOW...
3    trace.to_checkoutservice.duration_mean                      75                4                4                8  2024-01-17 07:01:00+00:00   0.003616    0.619875  2024-01-17T07:01:00+00:00|checkoutservice|0.00...
4     trace.to_checkoutservice.duration_p95                      75                4                4                8  2024-01-17 07:01:00+00:00   0.005657    1.635244  2024-01-17T07:01:00+00:00|checkoutservice|0.00...
5        trace.to_checkoutservice.row_count                      75                4                3                7  2024-01-17 06:59:00+00:00  23.000000  260.700000  2024-01-17T06:59:00+00:00|frontendservice|6|BE...
6     trace.from_checkoutservice.error_rate                     125                0                0                0                        NaT   0.000000    0.000000                                                   
7       trace.to_checkoutservice.error_rate                      75                0                0                0                        NaT   0.000000    0.000000                                                   )```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing df_metric; ensure timestamp and value types
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Target component
component = 'checkoutservice'

# 1) Global thresholds (p95, p05) per kpi_name for checkoutservice using full metric.csv
thr = (
    df_metric[df_metric['cmdb_id'] == component]
    .groupby('kpi_name', as_index=False)
    .agg(
        p95=('value', lambda x: x.quantile(0.95)),
        p05=('value', lambda x: x.quantile(0.05))
    )
)

# 2) Filter metric rows to incident window and checkoutservice
window_start = pd.Timestamp('2024-01-17 06:56:45', tz='UTC')
window_end = pd.Timestamp('2024-01-17 07:26:45', tz='UTC')

df_win = df_metric[
    (df_metric['cmdb_id'] == component) &
    (df_metric['timestamp_dt'] >= window_start) &
    (df_metric['timestamp_dt'] <= window_end)
].copy()

# Merge thresholds into window rows
df_win = df_win.merge(thr, on='kpi_name', how='left')

# 3) Keep only anomaly rows (value > p95 OR value < p05)
# Determine direction and compute severity_percent
def compute_direction_severity(row):
    v = row['value']
    p95 = row['p95']
    p05 = row['p05']
    if pd.isna(v) or (pd.isna(p95) and pd.isna(p05)):
        return pd.Series([None, np.nan])
    # ABOVE
    if (not pd.isna(p95)) and (v > p95):
        # avoid division by zero
        if p95 == 0:
            sev = np.nan
        else:
            sev = (v - p95) / p95 * 100.0
        return pd.Series(['ABOVE', sev])
    # BELOW
    if (not pd.isna(p05)) and (v < p05):
        if p05 > 0:
            sev = (p05 - v) / p05 * 100.0
        else:
            sev = np.nan
        return pd.Series(['BELOW', sev])
    return pd.Series([None, np.nan])

df_win[['direction', 'severity_percent']] = df_win.apply(compute_direction_severity, axis=1)

anomalies = df_win[df_win['direction'].notna()].copy()

# 4) Prepare anomaly rows: timestamp (UTC), kpi_name, value, p05, p95, direction, severity_percent
anomalies_rows = (
    anomalies
    .loc[:, ['timestamp_dt', 'kpi_name', 'value', 'p05', 'p95', 'direction', 'severity_percent']]
    .sort_values('timestamp_dt', ascending=True)
    .head(50)
)
# round severity for compactness
anomalies_rows['severity_percent'] = anomalies_rows['severity_percent'].round(3)

# 5) Per-kpi summary for checkoutservice (include all checkoutservice KPIs seen in thresholds)
# total_points_in_window per kpi
pts_in_window = df_win.groupby('kpi_name', as_index=False).size().rename(columns={'size':'total_points_in_window'})

# total_anomalies, max_severity_percent, earliest_anomaly_timestamp per kpi
grp_anom = anomalies.groupby('kpi_name')

summary_anom = grp_anom.agg(
    total_anomalies=('direction', 'count'),
    max_severity_percent=('severity_percent', 'max'),
    earliest_anomaly_timestamp=('timestamp_dt', 'min')
).reset_index()

# merge to include KPIs that may have zero points/anomalies in window
summary = thr[['kpi_name']].merge(pts_in_window, on='kpi_name', how='left').merge(summary_anom, on='kpi_name', how='left')

# fill missing numeric fields with zeros/NaN appropriately
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['total_anomalies'] = summary['total_anomalies'].fillna(0).astype(int)
summary['max_severity_percent'] = summary['max_severity_percent'].round(3)
# earliest_anomaly_timestamp remains NaT if none

# Order summary rows by total_anomalies desc for readability
per_kpi_summary = summary.sort_values('total_anomalies', ascending=False).reset_index(drop=True)

# Final compact outputs: anomalies_rows (up to 50) and per_kpi_summary (all checkoutservice KPIs)
anomalies_rows, per_kpi_summary

```
Out[4]:
```
Summary — checkoutservice metric anomalies (UTC):

- Time window: 2024-01-17 06:56:45 → 2024-01-17 07:26:45.

- Overall: Five checkoutservice KPIs show anomalies in the window (cpu, latency-50, latency-90, mem, workload). Each of these KPIs has 25 sampled points in the window and 4 anomaly events detected (value outside the global p05–p95). socket had 25 points and 0 anomalies.

- Earliest anomaly times (per KPI):
  - cpu: earliest 2024-01-17T06:59:00Z (4 anomalies total)
  - workload: earliest 2024-01-17T06:59:00Z (4 anomalies)
  - mem: earliest 2024-01-17T07:02:00Z (4 anomalies)
  - latency-50: earliest 2024-01-17T07:03:00Z (4 anomalies)
  - latency-90: earliest 2024-01-17T07:03:00Z (4 anomalies)

- Severity (max percent deviation relative to threshold per KPI):
  - latency-50: max severity ≈ 24.346% (largest deviation; a BELOW event)
  - workload: max severity ≈ 10.361% (BELOW)
  - cpu: max severity ≈ 7.207% (BELOW)
  - latency-90: max severity ≈ 4.532% (BELOW)
  - mem: max severity ≈ 0.501% (small ABOVE deviations)
  - socket: no anomalies

- Nature of anomalies: mix of BELOW and ABOVE threshold events across the window (examples: cpu/workload had BELOW at ~06:59; later there are ABOVE events for mem, workload, cpu and large latency ABOVE events around 07:14–07:19).

Actionable next steps:
1. Focus investigation around ~06:59–07:03 UTC (the earliest anomaly cluster) and also look at later spikes (~07:14–07:19).
2. Correlate these metric anomalies with the checkoutservice trace anomalies (trace anomalies started ~06:59) and any logs to determine whether the service experienced traffic shifts, resource starvation, or downstream latency causing the observed metric deviations.
3. Inspect host/container-level metrics and request traces for the exact timestamps with the highest severity (e.g., latency-50 at ~07:03 and 07:14) to pinpoint root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(                 timestamp_dt    kpi_name         value           p05           p95 direction  severity_percent
0   2024-01-17 06:59:00+00:00         cpu  3.202487e-01  3.451207e-01  4.609336e-01     BELOW             7.207
5   2024-01-17 06:59:00+00:00    workload  7.022000e-01  7.041967e-01  1.089620e+00     BELOW             0.284
21  2024-01-17 07:02:00+00:00         mem  1.190892e+07  1.192098e+07  1.216699e+07     BELOW             0.101
23  2024-01-17 07:02:00+00:00    workload  6.312333e-01  7.041967e-01  1.089620e+00     BELOW            10.361
25  2024-01-17 07:03:00+00:00  latency-50  9.790621e-02  1.294136e-01  4.579993e-01     BELOW            24.346
26  2024-01-17 07:03:00+00:00  latency-90  2.177983e-01  2.281387e-01  2.053321e+00     BELOW             4.532
32  2024-01-17 07:04:00+00:00  latency-90  2.279436e-01  2.281387e-01  2.053321e+00     BELOW             0.086
39  2024-01-17 07:05:00+00:00         mem  1.217140e+07  1.192098e+07  1.216699e+07     ABOVE             0.036
47  2024-01-17 07:06:00+00:00    workload  1.095633e+00  7.041967e-01  1.089620e+00     ABOVE             0.552
67  2024-01-17 07:10:00+00:00  latency-50  1.282538e-01  1.294136e-01  4.579993e-01     BELOW             0.896
75  2024-01-17 07:11:00+00:00         mem  1.191834e+07  1.192098e+07  1.216699e+07     BELOW             0.022
91  2024-01-17 07:14:00+00:00  latency-50  4.606250e-01  1.294136e-01  4.579993e-01     ABOVE             0.573
92  2024-01-17 07:14:00+00:00  latency-90  2.082776e+00  2.281387e-01  2.053321e+00     ABOVE             1.435
108 2024-01-17 07:17:00+00:00         cpu  3.448877e-01  3.451207e-01  4.609336e-01     BELOW             0.068
115 2024-01-17 07:18:00+00:00  latency-50  4.638889e-01  1.294136e-01  4.579993e-01     ABOVE             1.286
120 2024-01-17 07:19:00+00:00         cpu  4.806726e-01  3.451207e-01  4.609336e-01     ABOVE             4.282
125 2024-01-17 07:19:00+00:00    workload  1.147783e+00  7.041967e-01  1.089620e+00     ABOVE             5.338
134 2024-01-17 07:21:00+00:00  latency-90  2.068895e+00  2.281387e-01  2.053321e+00     ABOVE             0.759
138 2024-01-17 07:22:00+00:00         cpu  4.619499e-01  3.451207e-01  4.609336e-01     ABOVE             0.221
147 2024-01-17 07:23:00+00:00         mem  1.222798e+07  1.192098e+07  1.216699e+07     ABOVE             0.501,      kpi_name  total_points_in_window  total_anomalies  max_severity_percent earliest_anomaly_timestamp
0         cpu                      25                4                 7.207  2024-01-17 06:59:00+00:00
1  latency-50                      25                4                24.346  2024-01-17 07:03:00+00:00
2  latency-90                      25                4                 4.532  2024-01-17 07:03:00+00:00
3         mem                      25                4                 0.501  2024-01-17 07:02:00+00:00
4    workload                      25                4                10.361  2024-01-17 06:59:00+00:00
5      socket                      25                0                   NaN                        NaT)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp and value parsed correctly
df_trace['timestamp_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Identify trace_name values containing 'checkoutservice' (case-insensitive)
substr = 'checkoutservice'
mask = df_trace['trace_name'].str.contains(substr, case=False, na=False)
trace_names_found = sorted(df_trace.loc[mask, 'trace_name'].unique().tolist())

# Compute global p95 and p05 per identified trace_name using the full trace.csv
if trace_names_found:
    thresholds = (
        df_trace[df_trace['trace_name'].isin(trace_names_found)]
        .groupby('trace_name', as_index=False)
        .agg(
            p95=('value', lambda x: x.quantile(0.95)),
            p05=('value', lambda x: x.quantile(0.05))
        )
    )
else:
    thresholds = pd.DataFrame(columns=['trace_name','p05','p95'])

# 2) Filter trace.csv to the incident window and keep only checkoutservice trace rows
window_start = pd.Timestamp('2024-01-17 06:56:45', tz='UTC')
window_end = pd.Timestamp('2024-01-17 07:26:45', tz='UTC')

df_win = df_trace[
    (df_trace['timestamp_dt'] >= window_start) &
    (df_trace['timestamp_dt'] <= window_end) &
    (df_trace['trace_name'].isin(trace_names_found))
].copy()

# Merge thresholds (global) into windowed rows
df_win = pd.merge(df_win, thresholds, on='trace_name', how='left')

# 3) Identify anomalies and compute direction and severity_percent
# Vectorized calculations where possible
df_win['above'] = (df_win['value'] > df_win['p95'])
df_win['below'] = (df_win['value'] < df_win['p05'])

# direction
df_win['direction'] = np.where(df_win['above'], 'ABOVE', np.where(df_win['below'], 'BELOW', None))

# severity_percent
def compute_sev(row):
    v = row['value']
    if row['direction'] == 'ABOVE':
        p95 = row['p95']
        if pd.isna(p95) or p95 == 0:
            return np.nan
        return (v - p95) / p95 * 100.0
    if row['direction'] == 'BELOW':
        p05 = row['p05']
        if pd.isna(p05) or p05 == 0:
            return np.nan
        return (p05 - v) / p05 * 100.0
    return np.nan

df_win['severity_percent'] = df_win.apply(compute_sev, axis=1)

# 4) Keep only anomaly rows and prepare compact anomaly table
anomalies = df_win[df_win['direction'].notna()].copy()
anomalies = anomalies.sort_values('timestamp_dt', ascending=True)

anomaly_rows = anomalies.loc[:, ['timestamp_dt','trace_name','cmdb_id','value','p05','p95','direction','severity_percent']].head(50)
anomaly_rows['severity_percent'] = anomaly_rows['severity_percent'].round(3)

# 5) Per-trace summary for all checkoutservice trace groups
# total_points_in_window per trace_name
pts = df_win.groupby('trace_name', as_index=False).size().rename(columns={'size':'total_points_in_window'})

# anomaly summary per trace_name
if not anomalies.empty:
    an_grp = anomalies.groupby('trace_name', sort=False).agg(
        total_anomalies=('direction','count'),
        max_severity_percent=('severity_percent','max'),
        earliest_anomaly_timestamp=('timestamp_dt','min')
    ).reset_index()
else:
    an_grp = pd.DataFrame(columns=['trace_name','total_anomalies','max_severity_percent','earliest_anomaly_timestamp'])

# Merge to include trace groups with zero anomalies
per_trace_summary = pd.DataFrame({'trace_name': trace_names_found})
per_trace_summary = per_trace_summary.merge(pts, on='trace_name', how='left')
per_trace_summary = per_trace_summary.merge(an_grp, on='trace_name', how='left')

# Fill defaults for traces with no window points or no anomalies
per_trace_summary['total_points_in_window'] = per_trace_summary['total_points_in_window'].fillna(0).astype(int)
per_trace_summary['total_anomalies'] = per_trace_summary['total_anomalies'].fillna(0).astype(int)
per_trace_summary['max_severity_percent'] = per_trace_summary['max_severity_percent'].round(3)
# earliest_anomaly_timestamp stays NaT when none

# Sort summary by total_anomalies desc
per_trace_summary = per_trace_summary.sort_values('total_anomalies', ascending=False).reset_index(drop=True)

# Final compact outputs: anomaly_rows (up to 50) and per_trace_summary (all checkoutservice trace groups)
anomaly_rows, per_trace_summary

```
Out[5]:
```
Summary — checkoutservice-related traces (plain English, UTC):

- Trace groups found: trace.from_checkoutservice.* and trace.to_checkoutservice.* (8 series total, including duration_mean, duration_p95, row_count, error_rate).

- Global thresholds were computed from the full trace.csv. Error_rate series have zero thresholds (no signal).

- Incident-window anomalies (2024-01-17 06:56:45 → 07:26:45):
  - Highest anomaly volumes and earliest onset:
    - trace.from_checkoutservice.duration_mean: 125 points, 14 anomalies, earliest 2024-01-17 06:59:00, max severity ≈ 105.3%.
    - trace.from_checkoutservice.duration_p95: 125 points, 14 anomalies, earliest 06:59:00, max severity ≈ 318.5%.
    - trace.from_checkoutservice.row_count: 125 points, 12 anomalies, earliest 06:59:00, max severity = 70%.
  - to_checkoutservice traces also anomalous:
    - to_checkoutservice.duration_mean: 75 points, 8 anomalies, earliest 07:01:00, max severity ≈ 18.0%.
    - to_checkoutservice.duration_p95: 75 points, 8 anomalies, earliest 07:01:00, max severity ≈ 10.6%.
    - to_checkoutservice.row_count: 75 points, 7 anomalies, earliest 06:59:00, max severity ≈ 73.9%.
  - error_rate traces had no anomalies (all zeros).

- Notable anomaly patterns from sample rows:
  - Around 06:59 UTC many row_count values were well below historical p05 (e.g., emailservice/frontend/paymentservice row_count = 6 while p05 ~20–23) — large relative drops in observed request counts.
  - Around the same early window some duration metrics were BELOW tiny historical p05 values (very small absolute numbers but flagged relative to p05).
  - Later (e.g., ~07:18 UTC) there are clear ABOVE anomalies: large increases in duration_mean and duration_p95 for traffic involving frontendservice and checkoutservice (severity ~10–11%+ and some very large relative jumps for p95 series).

- Interpretation / recommended next steps:
  1. Two correlated phenomena: an early drop in row_count (traffic) beginning ~06:59 UTC, and later strong latency spikes (duration_mean/p95) peaking ~07:18 UTC. Both affect checkoutservice call paths (both incoming and outgoing traces).
  2. Correlate these trace anomalies with the checkoutservice metric anomalies (CPU, mem, workload, latency) previously observed around 06:59–07:19 to determine whether resource issues or upstream traffic changes caused the latency spikes.
  3. Inspect upstream callers listed in sample rows (frontendservice, productcatalogservice, emailservice, paymentservice) for concurrent issues and check service logs around 06:59 and 07:18 UTC.

The original code execution output of IPython Kernel is also provided below for reference:

(                 timestamp_dt                                trace_name                cmdb_id     value        p05         p95 direction  severity_percent
15  2024-01-17 06:59:00+00:00      trace.from_checkoutservice.row_count           emailservice  6.000000  20.000000  255.800000     BELOW            70.000
19  2024-01-17 06:59:00+00:00        trace.to_checkoutservice.row_count        frontendservice  6.000000  23.000000  260.700000     BELOW            73.913
23  2024-01-17 06:59:00+00:00      trace.from_checkoutservice.row_count         paymentservice  6.000000  20.000000  255.800000     BELOW            70.000
24  2024-01-17 06:59:00+00:00  trace.from_checkoutservice.duration_mean  productcatalogservice  0.000014   0.000015    0.066966     BELOW             9.263
25  2024-01-17 06:59:00+00:00   trace.from_checkoutservice.duration_p95  productcatalogservice  0.000020   0.000022    0.311075     BELOW             9.050
..                        ...                                       ...                    ...       ...        ...         ...       ...               ...
625 2024-01-17 07:18:00+00:00     trace.to_checkoutservice.duration_p95        frontendservice  1.808504   0.005657    1.635244     ABOVE            10.595
624 2024-01-17 07:18:00+00:00    trace.to_checkoutservice.duration_mean        frontendservice  0.689311   0.003616    0.619875     ABOVE            11.202
620 2024-01-17 07:18:00+00:00  trace.from_checkoutservice.duration_mean           emailservice  0.073868   0.000015    0.066966     ABOVE            10.306
609 2024-01-17 07:18:00+00:00   trace.from_checkoutservice.duration_p95        checkoutservice  0.318542   0.000022    0.311075     ABOVE             2.400
608 2024-01-17 07:18:00+00:00  trace.from_checkoutservice.duration_mean        checkoutservice  0.067475   0.000015    0.066966     ABOVE             0.759

[50 rows x 8 columns],                                  trace_name  total_points_in_window  total_anomalies  max_severity_percent earliest_anomaly_timestamp
0  trace.from_checkoutservice.duration_mean                     125               14               105.280  2024-01-17 06:59:00+00:00
1   trace.from_checkoutservice.duration_p95                     125               14               318.539  2024-01-17 06:59:00+00:00
2      trace.from_checkoutservice.row_count                     125               12                70.000  2024-01-17 06:59:00+00:00
3    trace.to_checkoutservice.duration_mean                      75                8                18.042  2024-01-17 07:01:00+00:00
4     trace.to_checkoutservice.duration_p95                      75                8                10.595  2024-01-17 07:01:00+00:00
5        trace.to_checkoutservice.row_count                      75                7                73.913  2024-01-17 06:59:00+00:00
6     trace.from_checkoutservice.error_rate                     125                0                   NaN                        NaT
7       trace.to_checkoutservice.error_rate                      75                0                   NaN                        NaT)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```